# Part 2: Text Analysis

## Objective

The objective is to analyze 50 climate and renewable-energy
articles collected during Part 1.

Two text analysis algorithms will be implemented:

1. TF-IDF with cosine similarity for topic relevance.
2. VADER for sentiment analysis.

The results will be combined to produce an importance score
with a positive or negative direction.

In [1]:
import nltk

nltk.download("vader_lexicon")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\300411418\AppData\Roaming\nltk_data...


True

In [2]:
#import analysis libraries
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from nltk.sentiment import SentimentIntensityAnalyzer

print("Text analysis libraries loaded successfully!")

Text analysis libraries loaded successfully!


In [3]:
#load scraped dataset
df = pd.read_csv("../data/scraped_articles.csv")

print("Articles loaded:", len(df))

display(df.head())

Articles loaded: 50


,topic,title,url,publication_date,article_text,word_count,extraction_status
0,Climate Change,Cop31 collapses: why did climate talks fall ap...,https://www.theguardian.com/news/audio/2026/no...,"Mon, 23 Nov 2026 17:19:55 GMT",Cop31 collapses: why did climate talks fall ap...,101,Success
1,Climate Change,Cranberry jam: warming winters and extreme wea...,https://www.theguardian.com/us-news/2026/nov/2...,"Mon, 23 Nov 2026 15:00:49 GMT","It is harvest time in rural Wisconsin, and the...",572,Success
2,Climate Change,Plant-based lunches could cut Australia’s emis...,https://www.theguardian.com/environment/2026/n...,"Mon, 23 Nov 2026 13:58:01 GMT",Australians could collectively cut climate pol...,604,Success
3,Climate Change,Failure of Cop31 prompts calls for urgent refo...,https://www.theguardian.com/environment/2026/n...,"Mon, 23 Nov 2026 12:21:23 GMT",The failure at Cop31 to reach a keystone agree...,853,Success
4,Climate Change,‘Difficult geopolitical times’: Albanese defen...,https://www.theguardian.com/australia-news/202...,"Mon, 23 Nov 2026 04:21:00 GMT",Anthony Albanese has defended Chris Bowen from...,950,Success


In [4]:
# Define our research topic
topic_description = """
Climate change global warming greenhouse gas emissions
carbon pollution renewable energy solar power wind energy
clean energy sustainability climate policy fossil fuels
energy transition environmental protection
"""

# Combine all article text and the research topic
documents = df["article_text"].fillna("").tolist()

all_text = documents + [topic_description]

# Create TF-IDF vectors
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(all_text)

# Calculate cosine similarity between each article and topic
topic_vector = tfidf_matrix[-1]

article_vectors = tfidf_matrix[:-1]

similarities = cosine_similarity(
    article_vectors,
    topic_vector
).flatten()

df["relevance_score"] = similarities

print("TF-IDF analysis completed!")

display(
    df[["title", "relevance_score"]]
    .sort_values("relevance_score", ascending=False)
    .head(10)
)

TF-IDF analysis completed!


,title,relevance_score
28,Who’s who at Cop31: key players at the climate...,0.251752
22,Slight fall in carbon emissions from fossil fu...,0.204506
31,The good news out of Cop31? That most of the w...,0.192240
6,Where do people want to see Britain’s climate ...,0.176846
23,Senate advances bill to speed up projects from...,0.171242
15,Passing bill to speed up gas projects and data...,0.171220
17,California governor Gavin Newsom endorses glob...,0.169884
13,China commits to low-carbon electrification an...,0.162898
16,Great Britain’s oldest renewables projects cou...,0.156828
36,The Guardian view on Cop31: a just transition ...,0.153495


### Sentiment Analysis Using VADER

VADER (Valence Aware Dictionary and sEntiment Reasoner) was used
to evaluate the sentiment of each collected article.

It produces positive, negative, neutral and compound sentiment
scores. The compound score ranges from -1 (most negative)
to +1 (most positive).

A limitation of VADER is that it evaluates the emotional tone
of language rather than the actual impact or position of an article.

In [5]:
from nltk.sentiment import SentimentIntensityAnalyzer

# Initialize VADER
sia = SentimentIntensityAnalyzer()

# Analyze each article
sentiment_results = df["article_text"].fillna("").apply(
    sia.polarity_scores
)

# Extract sentiment values
df["positive_sentiment"] = sentiment_results.apply(
    lambda x: x["pos"]
)

df["negative_sentiment"] = sentiment_results.apply(
    lambda x: x["neg"]
)

df["neutral_sentiment"] = sentiment_results.apply(
    lambda x: x["neu"]
)

df["compound_sentiment"] = sentiment_results.apply(
    lambda x: x["compound"]
)

print("VADER sentiment analysis completed!")

VADER sentiment analysis completed!


In [6]:
display(
    df[
        [
            "title",
            "positive_sentiment",
            "negative_sentiment",
            "neutral_sentiment",
            "compound_sentiment"
        ]
    ].head(10)
)

,title,positive_sentiment,negative_sentiment,neutral_sentiment,compound_sentiment
0,Cop31 collapses: why did climate talks fall ap...,0.079,0.145,0.775,-0.8074
1,Cranberry jam: warming winters and extreme wea...,0.114,0.056,0.831,0.9869
2,Plant-based lunches could cut Australia’s emis...,0.058,0.023,0.919,0.9630
3,Failure of Cop31 prompts calls for urgent refo...,0.100,0.086,0.814,0.8059
4,‘Difficult geopolitical times’: Albanese defen...,0.102,0.087,0.811,0.9333
5,The Guardian view on climate and debt: a vicio...,0.055,0.169,0.776,-0.9959
6,Where do people want to see Britain’s climate ...,0.186,0.076,0.738,0.9984
7,How a tiny rare frog became an ecological warn...,0.082,0.077,0.840,0.6317
8,"‘Legally, they don’t exist’: the hybrid crabs ...",0.050,0.066,0.884,-0.9622
9,What’s left to do as Cop31 enters its frantic ...,0.111,0.127,0.763,-0.9699


In [7]:
#calculate importance score 
# Calculate importance score

df["importance_score"] = (
    100
    * df["relevance_score"]
    * df["compound_sentiment"]
)

print("Importance scores calculated successfully!")

Importance scores calculated successfully!


In [8]:
#view importance scores
display(
    df[
        [
            "title",
            "relevance_score",
            "compound_sentiment",
            "importance_score"
        ]
    ].head(10)
)

,title,relevance_score,compound_sentiment,importance_score
0,Cop31 collapses: why did climate talks fall ap...,0.076279,-0.8074,-6.158799
1,Cranberry jam: warming winters and extreme wea...,0.032687,0.9869,3.225857
2,Plant-based lunches could cut Australia’s emis...,0.089999,0.9630,8.666951
3,Failure of Cop31 prompts calls for urgent refo...,0.070459,0.8059,5.678276
4,‘Difficult geopolitical times’: Albanese defen...,0.126615,0.9333,11.816948
5,The Guardian view on climate and debt: a vicio...,0.099966,-0.9959,-9.955568
6,Where do people want to see Britain’s climate ...,0.176846,0.9984,17.656286
7,How a tiny rare frog became an ecological warn...,0.009336,0.6317,0.589742
8,"‘Legally, they don’t exist’: the hybrid crabs ...",0.002617,-0.9622,-0.251854
9,What’s left to do as Cop31 enters its frantic ...,0.113134,-0.9699,-10.972914


In [9]:
#classify the importance direction
def classify_direction(score):

    if score > 2:
        return "Positive"

    elif score < -2:
        return "Negative"

    else:
        return "Neutral"


df["direction"] = df["importance_score"].apply(
    classify_direction
)

print("Importance direction classified successfully!")

Importance direction classified successfully!


In [10]:
#confirm the direction distribution 
print("Article Direction Distribution")
print("------------------------------")

print(df["direction"].value_counts())

print("\nPercentage Distribution")

print(
    (df["direction"].value_counts(normalize=True) * 100)
    .round(2)
)

Article Direction Distribution
------------------------------
direction
Positive    29
Neutral     11
Negative    10
Name: count, dtype: int64

Percentage Distribution
direction
Positive    58.0
Neutral     22.0
Negative    20.0
Name: proportion, dtype: float64


* 58% of articles received positive importance scores.

* 22% received neutral scores.

* 20% received negative scores.

In [14]:
#generate article summaries
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer


def generate_summary(text, num_sentences=3):

    sentences = re.split(
        r'(?<=[.!?])\s+',
        str(text)
    )

    unwanted = [
        "the latest",
        "read more",
        "sign up",
        "advertisement"
    ]

    sentences = [
        s.strip() for s in sentences
        if len(s.split()) >= 8
        and s.strip().lower() not in unwanted
    ]

    if not sentences:
        return ""

    if len(sentences) <= num_sentences:
        return " ".join(sentences)

    vectorizer = TfidfVectorizer(
        stop_words="english"
    )

    try:
        matrix = vectorizer.fit_transform(sentences)

        scores = np.asarray(
            matrix.mean(axis=1)
        ).flatten()

        top_indices = np.argsort(scores)[-num_sentences:]

        top_indices = sorted(top_indices)

        return " ".join(
            sentences[i] for i in top_indices
        )

    except ValueError:
        return " ".join(sentences[:num_sentences])

In [15]:
df["summary"] = df["article_text"].apply(
    generate_summary
)

print("Article summaries generated successfully!")
print("Total summaries:", df["summary"].notna().sum())

Article summaries generated successfully!
Total summaries: 50


In [16]:
#check article summaries 
display(
    df[
        [
            "title",
            "summary"
        ]
    ].head(5)
)

,title,summary
0,Cop31 collapses: why did climate talks fall ap...,- The Latest\nCountries at the Cop31 climate s...
1,Cranberry jam: warming winters and extreme wea...,Shifting seasons and unpredictable extremes\nI...
2,Plant-based lunches could cut Australia’s emis...,A nationwide shift to plant-based lunches coul...
3,Failure of Cop31 prompts calls for urgent refo...,We need a structural reform that delivers on c...
4,‘Difficult geopolitical times’: Albanese defen...,"Bill Hare, the Australian chief executive of C..."


In [17]:
df["summary"] = df["summary"].str.replace(
    r"(?i)^[-–\s]*the latest\s*",
    "",
    regex=True
)

display(df[["title", "summary"]].head(5))

,title,summary
0,Cop31 collapses: why did climate talks fall ap...,Countries at the Cop31 climate summit failed t...
1,Cranberry jam: warming winters and extreme wea...,Shifting seasons and unpredictable extremes\nI...
2,Plant-based lunches could cut Australia’s emis...,A nationwide shift to plant-based lunches coul...
3,Failure of Cop31 prompts calls for urgent refo...,We need a structural reform that delivers on c...
4,‘Difficult geopolitical times’: Albanese defen...,"Bill Hare, the Australian chief executive of C..."


In [18]:
#save final analysis results 
final_results = df[
    [
        "url",
        "title",
        "topic",
        "publication_date",
        "summary",
        "relevance_score",
        "compound_sentiment",
        "importance_score",
        "direction",
        "word_count"
    ]
].copy()

# Sort by importance score
final_results = final_results.sort_values(
    "importance_score",
    ascending=False
)

print("Final dataset created!")
print("Total articles:", len(final_results))

display(final_results.head(10))

Final dataset created!
Total articles: 50


,url,title,topic,publication_date,summary,relevance_score,compound_sentiment,importance_score,direction,word_count
28,https://www.theguardian.com/environment/2026/n...,Who’s who at Cop31: key players at the climate...,Renewable Energy,"Sun, 08 Nov 2026 11:48:56 GMT",Brazil\nThe recently re-elected President Lula...,0.251752,0.9963,25.082088,Positive,1694
31,https://www.theguardian.com/environment/commen...,The good news out of Cop31? That most of the w...,Environment,"Sun, 22 Nov 2026 22:30:48 GMT",Off the coast of Western Australia there’s the...,0.192240,0.9475,18.214770,Positive,949
6,https://www.theguardian.com/commentisfree/2026...,Where do people want to see Britain’s climate ...,Climate Change,"Tue, 17 Nov 2026 13:32:44 GMT",Early predictions by the Met Office indicate t...,0.176846,0.9984,17.656286,Positive,792
22,https://www.theguardian.com/environment/2026/n...,Slight fall in carbon emissions from fossil fu...,Renewable Energy,"Tue, 10 Nov 2026 11:55:06 GMT","Accelerated deployment of solar, wind, batteri...",0.204506,0.8429,17.237850,Positive,885
23,https://www.theguardian.com/us-news/2026/nov/0...,Senate advances bill to speed up projects from...,Renewable Energy,"Tue, 10 Nov 2026 02:28:00 GMT","House minority leader Hakeem Jeffries, expecte...",0.171242,0.9996,17.117324,Positive,1654
17,https://www.theguardian.com/environment/2026/n...,California governor Gavin Newsom endorses glob...,Renewable Energy,"Fri, 13 Nov 2026 16:31:28 GMT",Speaking at the Cop31 UN climate summit on Thu...,0.169884,0.9947,16.898343,Positive,920
15,https://www.theguardian.com/us-news/2026/nov/1...,Passing bill to speed up gas projects and data...,Renewable Energy,"Mon, 16 Nov 2026 14:34:17 GMT","The legislation, known as the Bipartisan Ameri...",0.171220,0.9852,16.868558,Positive,884
13,https://www.theguardian.com/environment/2026/n...,China commits to low-carbon electrification an...,Renewable Energy,"Sat, 21 Nov 2026 09:47:57 GMT",China has significantly reduced the cost of cl...,0.162898,0.9978,16.253998,Positive,928
16,https://www.theguardian.com/business/2026/nov/...,Great Britain’s oldest renewables projects cou...,Renewable Energy,"Mon, 16 Nov 2026 11:56:20 GMT",The new private law contracts could also offer...,0.156828,0.9931,15.574610,Positive,760
21,https://www.theguardian.com/environment/2026/n...,US and rest of world must follow China’s lead ...,Renewable Energy,"Tue, 10 Nov 2026 12:08:23 GMT",Li Shuo from the Asia Society Policy Institute...,0.148182,0.9970,14.773761,Positive,1192


In [19]:
#save final csv file 
final_results.to_csv(
    "../data/analysis_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV file saved successfully!")

CSV file saved successfully!
